In [ ]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
import logging
import time

class GoogleSheetSync:
    def __init__(self, credentials_file: str):
        # Set up logging
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('sheet_sync.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
        
        # Initialize the Google Sheets API
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        self.creds = service_account.Credentials.from_service_account_file(
            credentials_file, scopes=scopes)
        self.service = build('sheets', 'v4', credentials=self.creds)
    
    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        """Extract the spreadsheet ID from a Google Sheets URL"""
        import re
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError("Invalid Google Sheet URL")
    
    def get_sheet_data(self, spreadsheet_id: str, sheet_range: str):
        """Get data from a specific range in a sheet"""
        result = self.service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id, range=sheet_range).execute()
        return result.get('values', [])
    
    def update_row(self, spreadsheet_id: str, range_name: str, row_data):
        """Update a single row in the spreadsheet"""
        body = {
            'values': [row_data]
        }
        result = self.service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id, range=range_name,
            valueInputOption='RAW', body=body).execute()
        return result
    
    def find_row_by_email(self, sheet1_data, email_column_index, email_to_find):
        """Find the row number containing the specified email"""
        for i, row in enumerate(sheet1_data):
            if len(row) > email_column_index and row[email_column_index] == email_to_find:
                return i + 1  # +1 because Google Sheets is 1-indexed
        return None
    
    def sync_all_rows(self, spreadsheet_id: str, email_column_index: int = 2):
        """
        Reads all rows from Sheet3, finds matching emails in Sheet1,
        and replaces each corresponding row in Sheet1 with data from Sheet3
        """
        try:
            # Get all data from Sheet3
            source_data = self.get_sheet_data(spreadsheet_id, "Sheet3!A:Z")
            
            if not source_data or len(source_data) <= 1:  # Check if there's data beyond header row
                self.logger.error("No data found in Sheet3 or only header row exists")
                return False
            
            # Get all data from Sheet1 to search for emails (more efficient than querying for each row)
            sheet1_data = self.get_sheet_data(spreadsheet_id, "Sheet1!A:Z")
            
            if not sheet1_data:
                self.logger.error("No data found in Sheet1")
                return False
            
            # Skip header row, start processing from row 2
            total_updated = 0
            total_failures = 0
            
            for i, source_row in enumerate(source_data[1:], start=2):  # Start from row 2 (index 1 + 2)
                try:
                    # Make sure email column exists in the source row
                    if len(source_row) <= email_column_index:
                        self.logger.warning(f"Row {i}: Email column index {email_column_index} out of range, skipping")
                        total_failures += 1
                        continue
                    
                    email_to_find = source_row[email_column_index]
                    self.logger.info(f"Processing row {i} with email: {email_to_find}")
                    
                    # Find the row in Sheet1 with the matching email
                    target_row_number = self.find_row_by_email(sheet1_data, email_column_index, email_to_find)
                    
                    if not target_row_number:
                        self.logger.warning(f"Row {i}: Email {email_to_find} not found in Sheet1, skipping")
                        total_failures += 1
                        continue
                    
                    # Update the row in Sheet1 with data from Sheet3
                    target_range = f"Sheet1!A{target_row_number}:Z{target_row_number}"
                    self.update_row(spreadsheet_id, target_range, source_row)
                    
                    self.logger.info(f"Successfully updated row {target_row_number} in Sheet1 with data from row {i} in Sheet3")
                    total_updated += 1
                    
                    # Add a small delay to avoid rate limiting
                    time.sleep(0.5)
                    
                except Exception as e:
                    self.logger.error(f"Error processing row {i}: {str(e)}")
                    total_failures += 1
            
            self.logger.info(f"Sync completed. Updated: {total_updated}, Failed: {total_failures}")
            return True
            
        except Exception as e:
            self.logger.error(f"Error in sync_all_rows: {str(e)}")
            return False

In [ ]:
def main():
    CREDENTIALS_FILE = "data/url-to-email-445616-cebe4868914f.json"
    EMAIL_COLUMN_INDEX = 4
    GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1zhDMUVl-75GiY-gIFc-jL9A2E3KVd9XpfV21D3pMVXo/edit?gid=855023356#gid=855023356" 
    
    syncer = GoogleSheetSync(CREDENTIALS_FILE)
    try:
        spreadsheet_id = syncer.extract_spreadsheet_id(GOOGLE_SHEET_URL)
        print(f"Starting to sync all rows from Sheet3 to Sheet1...")
        result = syncer.sync_all_rows(spreadsheet_id, EMAIL_COLUMN_INDEX)
        if result:
            print("Sync process completed successfully!")
        else:
            print("Sync process encountered errors. Check the logs for details.")
    except Exception as e:
        print(f"Error: {str(e)}")

In [ ]:
await main()